In [ ]:
!pip install transformers datasets sentence-transformers faiss-cpu \
            gradio wordcloud matplotlib seaborn pandas scikit-learn \
            plotly requests numpy -q
print("✅ All packages installed")

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import random

random.seed(42)
np.random.seed(42)

print("Loading Amazon Polarity dataset (2000 reviews)...")
# FIX: Use namespaced dataset ID; remove trust_remote_code (no longer supported)
dataset = load_dataset("fancyzhx/amazon_polarity", split="test[:2000]")
df = pd.DataFrame(dataset)

# Columns ('label', 'title', 'content') already exist — no reassignment needed

df['sentiment']  = df['label'].map({1: 'POSITIVE', 0: 'NEGATIVE'})
df['text']       = df['title'].fillna("") + ". " + df['content'].fillna("")
df['word_count'] = df['content'].fillna("").apply(lambda x: len(x.split()))
df['char_count'] = df['content'].fillna("").apply(len)
df['title_len']  = df['title'].fillna("").apply(len)

CATEGORIES = ['Electronics', 'Books', 'Clothing', 'Home & Kitchen', 'Sports', 'Beauty']
df['category'] = [random.choice(CATEGORIES) for _ in range(len(df))]

print(f"✅ {len(df)} reviews loaded")
print(f"Sentiment split: {df['sentiment'].value_counts().to_dict()}")
print(f"Categories: {df['category'].value_counts().to_dict()}")
df[['title', 'sentiment', 'category', 'word_count']].head(4)

In [ ]:
from transformers import pipeline
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("Loading DistilBERT sentiment model...")
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True,
    max_length=512
)

# Evaluate on 500 samples
print("Evaluating on 500 reviews...")
eval_df = df[:500].copy()
eval_results = sentiment_model(eval_df['text'].tolist(), batch_size=32)
eval_df['pred']       = [r['label'] for r in eval_results]
eval_df['confidence'] = [r['score'] for r in eval_results]

label_map = {'POSITIVE': 1, 'NEGATIVE': 0}
eval_df['pred_int'] = eval_df['pred'].map(label_map)

print("\n📊 Classification Report:")
print(classification_report(eval_df['label'], eval_df['pred_int'],
                             target_names=['Negative','Positive']))

# Confusion matrix
cm = confusion_matrix(eval_df['label'], eval_df['pred_int'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Neg','Predicted Pos'],
            yticklabels=['Actual Neg','Actual Pos'])
plt.title('DistilBERT Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Run on ALL 2000 reviews
print("\nRunning predictions on all 2000 reviews...")
all_results = sentiment_model(df['text'].tolist(), batch_size=32)
df['pred_sentiment'] = [r['label'] for r in all_results]
df['confidence']     = [r['score'] for r in all_results]
print(f"✅ Done. Avg confidence: {df['confidence'].mean():.2%}")

In [ ]:
from wordcloud import WordCloud
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ── Static matplotlib dashboard ──
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Amazon Review Intelligence — Analytics Dashboard',
             fontsize=16, fontweight='bold')

# 1. Sentiment donut
counts = df['pred_sentiment'].value_counts()
axes[0,0].pie(counts, labels=counts.index, autopct='%1.1f%%',
              colors=['#27ae60','#e74c3c'], startangle=90,
              wedgeprops={'width':0.55})
axes[0,0].set_title('Sentiment Distribution', fontweight='bold')

# 2. Positive % by category
cat_pct = df.groupby('category').apply(
    lambda x: (x['pred_sentiment']=='POSITIVE').mean()*100).sort_values()
cat_pct.plot(kind='barh', ax=axes[0,1], color='#27ae60', edgecolor='white')
axes[0,1].set_title('% Positive by Category', fontweight='bold')
axes[0,1].axvline(50, color='gray', linestyle='--', alpha=0.5)
axes[0,1].set_xlabel('% Positive')

# 3. Confidence distribution
axes[0,2].hist(df['confidence'], bins=30, color='#3498db', edgecolor='white', alpha=0.85)
axes[0,2].axvline(df['confidence'].mean(), color='red', linestyle='--',
                   label=f"Mean: {df['confidence'].mean():.2%}")
axes[0,2].set_title('Prediction Confidence', fontweight='bold')
axes[0,2].set_xlabel('Confidence Score')
axes[0,2].legend()

# 4. Review length by sentiment
for sent, color, lbl in [('POSITIVE','#27ae60','Positive'),('NEGATIVE','#e74c3c','Negative')]:
    axes[1,0].hist(df[df['pred_sentiment']==sent]['word_count'],
                   bins=30, alpha=0.6, color=color, label=lbl)
axes[1,0].set_title('Review Length by Sentiment', fontweight='bold')
axes[1,0].set_xlabel('Word Count')
axes[1,0].legend()

# 5. Positive word cloud
pos_text = " ".join(df[df['pred_sentiment']=='POSITIVE']['text'].tolist()[:400])
axes[1,1].imshow(WordCloud(width=600,height=300,background_color='white',
                            colormap='Greens',max_words=80).generate(pos_text))
axes[1,1].set_title('Positive Reviews — Key Words', fontweight='bold')
axes[1,1].axis('off')

# 6. Negative word cloud
neg_text = " ".join(df[df['pred_sentiment']=='NEGATIVE']['text'].tolist()[:400])
axes[1,2].imshow(WordCloud(width=600,height=300,background_color='white',
                            colormap='Reds',max_words=80).generate(neg_text))
axes[1,2].set_title('Negative Reviews — Key Words', fontweight='bold')
axes[1,2].axis('off')

plt.tight_layout()
plt.savefig('dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ dashboard.png saved")

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import pickle

print("Loading sentence embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating embeddings for 2000 reviews (~2 min with GPU)...")
texts = df['text'].tolist()
embeddings = embedder.encode(texts, batch_size=64, show_progress_bar=True)
embeddings = np.array(embeddings, dtype='float32')

# Normalize for cosine similarity
faiss.normalize_L2(embeddings)

# Build flat inner-product index
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

# Save everything
with open('review_index.pkl', 'wb') as f:
    pickle.dump({'index': index, 'texts': texts, 'df': df}, f)

print(f"✅ FAISS index built — {index.ntotal} vectors, dim={embeddings.shape[1]}")

In [ ]:
import requests
import json
import re

# ── CONFIG ───────────────────────────────────────────────────
ANTHROPIC_API_KEY = "sk-ant-..."   # 🔑 PASTE YOUR KEY HERE
MODEL             = "claude-haiku-4-5-20251001"

def call_claude(prompt, system="You are an expert Amazon review analyst.",
                max_tokens=700):
    """Core Claude API call — used by all functions below."""
    resp = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers={
            "x-api-key":         ANTHROPIC_API_KEY,
            "anthropic-version": "2023-06-01",
            "content-type":      "application/json"
        },
        json={
            "model":      MODEL,
            "max_tokens": max_tokens,
            "system":     system,
            "messages":   [{"role": "user", "content": prompt}]
        },
        timeout=30
    )
    if resp.status_code == 200:
        return resp.json()["content"][0]["text"]
    return f"[API Error {resp.status_code}: {resp.json().get('error',{}).get('message','')}]"


def parse_json_safe(text):
    """Strip markdown fences and parse JSON safely."""
    clean = re.sub(r'```json|```', '', text).strip()
    try:
        return json.loads(clean)
    except Exception:
        return {"error": "parse_failed", "raw": text[:200]}


# ── 1. RETRIEVAL ─────────────────────────────────────────────
def retrieve(query, top_k=6, category_filter=None):
    q_emb = embedder.encode([query], show_progress_bar=False).astype('float32')
    faiss.normalize_L2(q_emb)
    scores, idxs = index.search(q_emb, top_k * 4)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        row = df.iloc[idx]
        if category_filter and row['category'] != category_filter:
            continue
        results.append({
            'text':       row['text'][:400],
            'sentiment':  row['pred_sentiment'],
            'category':   row['category'],
            'confidence': float(row['confidence']),
            'relevance':  float(score)
        })
        if len(results) == top_k:
            break
    return results


# ── 2. RAG ANSWER ────────────────────────────────────────────
def rag_answer(query, category_filter=None):
    retrieved = retrieve(query, top_k=6, category_filter=category_filter)
    if not retrieved:
        return "No relevant reviews found.", [], 0, 0
    context = ""
    for i, r in enumerate(retrieved):
        context += (f"[Review {i+1} | {r['sentiment']} | "
                    f"{r['category']} | relevance:{r['relevance']:.2f}]\n"
                    f"{r['text']}\n\n")
    cat_note = f" Focus specifically on {category_filter}." if category_filter else ""
    prompt = (f"Answer using ONLY the reviews below.{cat_note} "
              f"Be specific and cite review numbers.\n\n"
              f"--- REVIEWS ---\n{context}--- END ---\n\n"
              f"Question: {query}")
    answer = call_claude(prompt)
    pos = sum(1 for r in retrieved if r['sentiment'] == 'POSITIVE')
    return answer, retrieved, pos, len(retrieved) - pos


# ── 3. SENTIMENT EXPLANATION ─────────────────────────────────
def explain_sentiment(review_text):
    result = sentiment_model(review_text[:512])[0]
    label, score = result['label'], result['score']
    explanation = call_claude(
        f"Review: {review_text[:400]}\n\n"
        f"Classified as {label} with {score:.0%} confidence.\n\n"
        f"Give 3 bullet points identifying the exact words or phrases "
        f"that drove this classification.\n"
        f"Format: • [word/phrase]: [reason it signals {label}]",
        system="You are a sentiment analysis expert. Be precise and brief."
    )
    return label, score, explanation


# ── 4. FAKE REVIEW DETECTION ─────────────────────────────────
def fake_review_score(review_text):
    prompt = (f"Analyze this Amazon review for authenticity.\n\n"
              f"Review: {review_text[:400]}\n\n"
              f"Score each red flag 0-10 (10=most suspicious):\n"
              f"1. generic_praise: no specific product details\n"
              f"2. grammar_flags: unusual or non-native phrasing\n"
              f"3. extreme_sentiment: no nuance at all\n"
              f"4. no_use_experience: reviewer never actually used it\n"
              f"5. promotional_tone: reads like marketing copy\n\n"
              f"Respond in JSON ONLY (no extra text):\n"
              f'{{"generic_praise":N,"grammar_flags":N,"extreme_sentiment":N,'
              f'"no_use_experience":N,"promotional_tone":N,'
              f'"overall_fake_score":N,"verdict":"LIKELY_FAKE|SUSPICIOUS|AUTHENTIC",'
              f'"reasoning":"one sentence"}}')
    return parse_json_safe(call_claude(
        prompt,
        system="You are a fraud detection expert. Respond only in valid JSON.",
        max_tokens=300
    ))


# ── 5. ASPECT-BASED SENTIMENT (ABSA) ─────────────────────────
def aspect_sentiment(review_text):
    prompt = (f"Extract aspect-level sentiment from this Amazon review.\n\n"
              f"Review: {review_text[:500]}\n\n"
              f"For each product aspect actually mentioned, give:\n"
              f"- aspect name\n- sentiment: POSITIVE, NEGATIVE, or NEUTRAL\n"
              f"- short evidence quote\n\n"
              f"Respond in JSON ONLY:\n"
              f'{{"aspects":['
              f'{{"aspect":"battery life","sentiment":"POSITIVE","evidence":"lasts all day"}}]}}')
    return parse_json_safe(call_claude(
        prompt,
        system="You are an aspect-based sentiment expert. Respond only in valid JSON.",
        max_tokens=400
    ))


# ── 6. HELPFULNESS PREDICTION ─────────────────────────────────
def predict_helpfulness(review_text):
    prompt = (f"Predict how helpful this Amazon review is to a potential buyer (0-100).\n\n"
              f"Review: {review_text[:400]}\n\n"
              f"Score these dimensions 0-100:\n"
              f"- specificity: mentions specific product details\n"
              f"- use_experience: describes actual usage\n"
              f"- balanced: covers pros AND cons\n"
              f"- length_quality: appropriate detail, not too short/long\n"
              f"- decision_impact: would help someone decide to buy\n\n"
              f"Respond in JSON ONLY:\n"
              f'{{"specificity":N,"use_experience":N,"balanced":N,'
              f'"length_quality":N,"decision_impact":N,"overall_helpfulness":N,'
              f'"summary":"one sentence"}}')
    return parse_json_safe(call_claude(
        prompt,
        system="You are a helpfulness prediction expert. Respond only in valid JSON.",
        max_tokens=300
    ))


# ── 7. PURCHASE INTENT SCORING ────────────────────────────────
def purchase_intent_signal(review_text):
    prompt = (f"Analyze this Amazon review for purchase conversion signals.\n\n"
              f"Review: {review_text[:400]}\n\n"
              f"Score 0-100:\n"
              f"- conversion_power: would make undecided buyer purchase\n"
              f"- concern_resolution: addresses common buyer fears\n"
              f"- competitive_signal: mentions alternatives or comparisons\n"
              f"- loyalty_signal: mentions repeat purchase or long-term use\n\n"
              f"Respond in JSON ONLY:\n"
              f'{{"conversion_power":N,"concern_resolution":N,'
              f'"competitive_signal":N,"loyalty_signal":N,'
              f'"key_phrase":"most conversion-driving phrase",'
              f'"overall_intent_score":N}}')
    return parse_json_safe(call_claude(
        prompt,
        system="You are a conversion optimization expert. Respond only in valid JSON.",
        max_tokens=300
    ))


# ── 8. CATEGORY COMPARISON ────────────────────────────────────
def compare_categories(cat_a, cat_b, aspect="overall quality and satisfaction"):
    def stats(cat):
        sub = df[df['category'] == cat]
        pos_pct    = (sub['pred_sentiment'] == 'POSITIVE').mean() * 100
        avg_conf   = sub['confidence'].mean() * 100
        avg_len    = sub['word_count'].mean()
        sample_pos = sub[sub['pred_sentiment']=='POSITIVE']['text'].iloc[0][:200] \
                     if len(sub[sub['pred_sentiment']=='POSITIVE']) else "N/A"
        sample_neg = sub[sub['pred_sentiment']=='NEGATIVE']['text'].iloc[0][:200] \
                     if len(sub[sub['pred_sentiment']=='NEGATIVE']) else "N/A"
        return pos_pct, avg_conf, avg_len, sample_pos, sample_neg

    pa, ca, la, spa, sna = stats(cat_a)
    pb, cb, lb, spb, snb = stats(cat_b)

    prompt = (f"Compare two Amazon product categories on: {aspect}\n\n"
              f"CATEGORY A: {cat_a}\n"
              f"  Positive rate: {pa:.1f}% | Confidence: {ca:.1f}% | "
              f"Avg length: {la:.0f} words\n"
              f"  Sample positive: \"{spa}\"\n"
              f"  Sample negative: \"{sna}\"\n\n"
              f"CATEGORY B: {cat_b}\n"
              f"  Positive rate: {pb:.1f}% | Confidence: {cb:.1f}% | "
              f"Avg length: {lb:.0f} words\n"
              f"  Sample positive: \"{spb}\"\n"
              f"  Sample negative: \"{snb}\"\n\n"
              f"Write a 5-point structured comparison:\n"
              f"1. Overall satisfaction winner (with % difference)\n"
              f"2. Top 2 strengths of {cat_a}\n"
              f"3. Top 2 strengths of {cat_b}\n"
              f"4. Main weakness of each\n"
              f"5. Buyer recommendation: who should choose which")
    analysis = call_claude(prompt, max_tokens=800)
    chart_data = {
        'categories':    [cat_a, cat_b],
        'positive_rate': [pa, pb],
        'avg_confidence':[ca, cb],
        'avg_length':    [la, lb]
    }
    return analysis, chart_data, pa, pb


# ── 9. TREND REPORT ──────────────────────────────────────────
def trend_analysis(category=None):
    sub = df[df['category'] == category] if category else df
    pos_pct   = (sub['pred_sentiment'] == 'POSITIVE').mean() * 100
    avg_conf  = sub['confidence'].mean() * 100
    hc_pos    = sub[(sub['pred_sentiment']=='POSITIVE') &
                    (sub['confidence']>0.95)]['text'].head(3).tolist()
    hc_neg    = sub[(sub['pred_sentiment']=='NEGATIVE') &
                    (sub['confidence']>0.95)]['text'].head(3).tolist()
    prompt = (f"Category: {category or 'All Products'}\n"
              f"Positive rate: {pos_pct:.1f}% | Avg confidence: {avg_conf:.1f}%\n\n"
              f"Most confident POSITIVE reviews:\n"
              f"{chr(10).join([f'- {t[:200]}' for t in hc_pos])}\n\n"
              f"Most confident NEGATIVE reviews:\n"
              f"{chr(10).join([f'- {t[:200]}' for t in hc_neg])}\n\n"
              f"Generate a concise trend report:\n"
              f"1. Health score (1-10) with justification\n"
              f"2. Top 3 recurring POSITIVE themes\n"
              f"3. Top 3 recurring NEGATIVE themes\n"
              f"4. One actionable product team recommendation\n"
              f"5. Risk level: Low / Medium / High — with reasoning")
    return call_claude(prompt, max_tokens=600)


print("✅ All 9 intelligence functions loaded and ready")

In [ ]:
import gradio as gr
from plotly.subplots import make_subplots
import plotly.graph_objects as go

ALL_CATS_OPTION = "All Categories"
CAT_OPTIONS     = [ALL_CATS_OPTION] + CATEGORIES


def make_comparison_chart(chart_data):
    fig = make_subplots(rows=1, cols=3,
        subplot_titles=('Positive Rate (%)', 'Avg Confidence (%)', 'Avg Word Count'))
    colors = ['#27ae60', '#3498db']
    for col, vals in [(1, chart_data['positive_rate']),
                      (2, chart_data['avg_confidence']),
                      (3, chart_data['avg_length'])]:
        fig.add_trace(go.Bar(x=chart_data['categories'],
                             y=[round(v, 1) for v in vals],
                             marker_color=colors, showlegend=False),
                      row=1, col=col)
    fig.update_layout(height=300,
                      margin=dict(t=40, b=10, l=10, r=10),
                      paper_bgcolor='rgba(0,0,0,0)',
                      plot_bgcolor='rgba(0,0,0,0)')
    return fig


def make_analytics_dashboard():
    stats = df.groupby('category').agg(
        positive_rate=('pred_sentiment', lambda x: (x=='POSITIVE').mean()*100),
        count=('text', 'count'),
        avg_conf=('confidence', lambda x: x.mean()*100)
    ).reset_index()

    fig = make_subplots(rows=2, cols=2,
        subplot_titles=('Positive Rate by Category', 'Review Volume',
                        'Overall Sentiment', 'Confidence vs Positivity'),
        specs=[[{'type':'bar'},   {'type':'pie'}],
               [{'type':'bar'},   {'type':'scatter'}]])

    fig.add_trace(go.Bar(x=stats['category'], y=stats['positive_rate'].round(1),
                         marker_color='#27ae60', name='Positive %'), row=1, col=1)
    fig.add_trace(go.Pie(labels=stats['category'], values=stats['count'],
                         hole=0.4, name='Volume'), row=1, col=2)
    sc = df['pred_sentiment'].value_counts()
    fig.add_trace(go.Bar(x=sc.index, y=sc.values,
                         marker_color=['#27ae60','#e74c3c']), row=2, col=1)
    fig.add_trace(go.Scatter(x=stats['avg_conf'], y=stats['positive_rate'],
                             mode='markers+text', text=stats['category'],
                             textposition='top center',
                             marker=dict(size=14, color='#3498db')), row=2, col=2)
    fig.update_layout(height=560, showlegend=False,
                      margin=dict(t=50, b=20, l=20, r=20),
                      paper_bgcolor='rgba(0,0,0,0)',
                      plot_bgcolor='rgba(0,0,0,0)')
    return fig


# ── Tab handlers ─────────────────────────────────────────────

def handle_sentiment(text):
    if not text.strip():
        return "⚠️ Please enter a review."
    label, score, expl = explain_sentiment(text)
    emoji = "✅" if label == "POSITIVE" else "❌"
    return f"{emoji} **{label}** — Confidence: `{score:.2%}`\n\n**Why Claude thinks so:**\n\n{expl}"


def handle_rag(query, cat_filter):
    if not query.strip():
        return "⚠️ Please enter a question.", ""
    cat = None if cat_filter == ALL_CATS_OPTION else cat_filter
    answer, retrieved, pos, neg = rag_answer(query, category_filter=cat)
    header  = f"**{len(retrieved)} reviews retrieved** ({pos} ✅ positive, {neg} ❌ negative)\n\n"
    sources = "\n\n---\n\n".join([
        f"**[{i+1}] {r['sentiment']} | {r['category']} | relevance: {r['relevance']:.2f}**\n\n{r['text'][:250]}..."
        for i, r in enumerate(retrieved)
    ])
    return header + "**Answer:**\n\n" + answer, sources


def handle_compare(cat_a, cat_b, aspect):
    if cat_a == cat_b:
        return "⚠️ Please select two different categories.", None
    analysis, chart_data, pa, pb = compare_categories(cat_a, cat_b, aspect)
    winner = cat_a if pa > pb else cat_b
    header = (f"### {cat_a} vs {cat_b}\n"
              f"Positive rates: **{cat_a}** `{pa:.1f}%` · **{cat_b}** `{pb:.1f}%` "
              f"→ 🏆 **{winner}** leads by `{abs(pa-pb):.1f}pp`\n\n")
    return header + analysis, make_comparison_chart(chart_data)


def handle_trend(cat):
    cat_arg = None if cat == ALL_CATS_OPTION else cat
    return trend_analysis(cat_arg)


def handle_advanced(text):
    if not text.strip():
        empty = {"error": "Please enter a review"}
        return empty, empty, empty, empty
    fake    = fake_review_score(text)
    absa    = aspect_sentiment(text)
    helpful = predict_helpfulness(text)
    intent  = purchase_intent_signal(text)
    return fake, absa, helpful, intent


def handle_multilingual(reviews_raw):
    if not reviews_raw.strip():
        return "⚠️ Please paste at least one review."
    lines = [l.strip() for l in reviews_raw.strip().split('\n') if l.strip()]
    prompt = (f"These are Amazon reviews from different global marketplaces, "
              f"possibly in different languages.\n\n"
              f"Reviews:\n" +
              "\n".join([f"- {r}" for r in lines[:8]]) +
              "\n\nFor each non-English review, translate it first.\n"
              f"Then produce ONE unified insight report:\n"
              f"1. Overall sentiment score (0-100)\n"
              f"2. Top 3 praise themes across all languages\n"
              f"3. Top 3 complaint themes across all languages\n"
              f"4. Which market/language has the most complaints\n"
              f"5. Cross-market recommendation for the product team")
    return call_claude(prompt, max_tokens=700)


# ── Build the app ─────────────────────────────────────────────
with gr.Blocks(title="Amazon Review Intelligence System",
               theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🛒 Amazon Review Intelligence System
    **DistilBERT** Sentiment &nbsp;·&nbsp; **FAISS** Vector Search &nbsp;·&nbsp;
    **Claude Haiku** LLM &nbsp;·&nbsp; **Fake Review Detection** &nbsp;·&nbsp;
    **ABSA** &nbsp;·&nbsp; **Plotly** Analytics
    ---
    """)

    with gr.Tabs():

        # ── TAB 1: Sentiment Analyzer ─────────────────────────
        with gr.Tab("📊 Sentiment Analyzer"):
            gr.Markdown("Classify any review and get Claude's line-by-line explanation.")
            with gr.Row():
                with gr.Column(scale=2):
                    t1_review = gr.Textbox(label="Paste Review", lines=6,
                        placeholder="Paste any Amazon review here...")
                    gr.Examples(inputs=t1_review, examples=[
                        ["This laptop is incredible. Battery lasts all day, screen is brilliant, totally silent. Best purchase this year."],
                        ["Complete garbage. Broke after 3 days, customer service ignored me. Avoid at all costs."],
                        ["It's okay. Does what it says but slightly overpriced for what you get."]
                    ])
                    t1_btn = gr.Button("Analyze with Claude", variant="primary")
                with gr.Column(scale=2):
                    t1_out = gr.Markdown(label="Result")
            t1_btn.click(handle_sentiment, inputs=t1_review, outputs=t1_out)

        # ── TAB 2: RAG Q&A ────────────────────────────────────
        with gr.Tab("🤖 Ask the Reviews"):
            gr.Markdown("Ask anything. Claude reads the most relevant reviews and answers with cited evidence.")
            with gr.Row():
                t2_query  = gr.Textbox(label="Your Question", lines=2, scale=3,
                    placeholder="e.g. What are the most common complaints?")
                t2_filter = gr.Dropdown(CAT_OPTIONS, value=ALL_CATS_OPTION,
                                        label="Category Filter", scale=1)
            gr.Examples(inputs=t2_query, examples=[
                ["What do customers love most about these products?"],
                ["What are the biggest quality complaints?"],
                ["Are there any shipping or delivery issues?"],
                ["What makes customers give 5-star reviews?"],
                ["Which problems cause customers to request refunds?"]
            ])
            t2_btn = gr.Button("Ask Claude", variant="primary")
            t2_ans = gr.Markdown(label="Claude's Answer")
            with gr.Accordion("View retrieved source reviews", open=False):
                t2_src = gr.Markdown()
            t2_btn.click(handle_rag, inputs=[t2_query, t2_filter],
                         outputs=[t2_ans, t2_src])

        # ── TAB 3: Category Comparison ────────────────────────
        with gr.Tab("⚖️ Compare Categories"):
            gr.Markdown("Head-to-head comparison. Claude analyzes sentiment, themes, and gives a verdict.")
            with gr.Row():
                t3_a      = gr.Dropdown(CATEGORIES, value=CATEGORIES[0], label="Category A")
                t3_b      = gr.Dropdown(CATEGORIES, value=CATEGORIES[1], label="Category B")
                t3_aspect = gr.Textbox(value="overall quality and customer satisfaction",
                                       label="Comparison Aspect")
            t3_btn   = gr.Button("Compare with Claude", variant="primary")
            t3_out   = gr.Markdown()
            t3_chart = gr.Plot(label="Head-to-Head Metrics")
            t3_btn.click(handle_compare, inputs=[t3_a, t3_b, t3_aspect],
                         outputs=[t3_out, t3_chart])

        # ── TAB 4: Trend Report ───────────────────────────────
        with gr.Tab("📈 Trend Report"):
            gr.Markdown("Claude generates a health score, recurring themes, and a product team recommendation.")
            t4_cat = gr.Dropdown(CAT_OPTIONS, value=ALL_CATS_OPTION, label="Category")
            t4_btn = gr.Button("Generate Report", variant="primary")
            t4_out = gr.Markdown()
            t4_btn.click(handle_trend, inputs=t4_cat, outputs=t4_out)

        # ── TAB 5: Analytics Dashboard ────────────────────────
        with gr.Tab("📉 Analytics Dashboard"):
            gr.Markdown("Interactive Plotly overview across all categories.")
            t5_btn   = gr.Button("Load Dashboard", variant="primary")
            t5_chart = gr.Plot()
            t5_btn.click(make_analytics_dashboard, inputs=[], outputs=t5_chart)

        # ── TAB 6: Advanced Intelligence ─────────────────────
        with gr.Tab("🔬 Advanced Intelligence"):
            gr.Markdown("""
            **4 capabilities Amazon doesn't have at scale:**
            fake review detection · aspect-based sentiment · helpfulness prediction · purchase intent
            """)
            t6_review = gr.Textbox(label="Paste Review", lines=5,
                placeholder="Paste any review to run all 4 analyses simultaneously...")
            gr.Examples(inputs=t6_review, examples=[
                ["AMAZING product!! Best thing I ever bought!! Five stars!!!! Will buy again!!!!"],
                ["The noise-cancelling headphones work brilliantly but the battery only lasts 12h instead of the advertised 20h. Build quality feels premium though."],
                ["I've tried 3 other brands and this is by far the best. My wife uses it daily and we've already ordered a second one for the kitchen."]
            ])
            t6_btn = gr.Button("Run Full Intelligence Analysis", variant="primary")
            with gr.Row():
                t6_fake    = gr.JSON(label="🕵️ Fake Review Score")
                t6_absa    = gr.JSON(label="📐 Aspect Sentiments")
            with gr.Row():
                t6_helpful = gr.JSON(label="📊 Helpfulness Prediction")
                t6_intent  = gr.JSON(label="🔮 Purchase Intent Signals")
            t6_btn.click(handle_advanced, inputs=t6_review,
                         outputs=[t6_fake, t6_absa, t6_helpful, t6_intent])

        # ── TAB 7: Cross-Lingual Aggregation ─────────────────
        with gr.Tab("🌍 Multilingual Aggregator"):
            gr.Markdown("""
            Paste reviews in any language (English, Hindi, German, Japanese, etc.).
            Claude translates and synthesizes them into one unified insight.
            """)
            t7_input = gr.Textbox(label="Reviews (one per line, any language)", lines=8,
                placeholder="Paste reviews here, one per line...\n\nExamples:\nGreat product!\nExcelente calidad, lo recomiendo.\nProduit de mauvaise qualité.")
            t7_btn   = gr.Button("Aggregate with Claude", variant="primary")
            t7_out   = gr.Markdown()
            t7_btn.click(handle_multilingual, inputs=t7_input, outputs=t7_out)

    gr.Markdown("""
    ---
    **Full pipeline:** Reviews → `DistilBERT` classify → `MiniLM-L6-v2` embed →
    `FAISS` retrieve → `Claude Haiku` generate + analyze
    **Models:** distilbert-base-uncased-finetuned-sst-2-english · all-MiniLM-L6-v2 ·
    claude-haiku-4-5 &nbsp;|&nbsp; **Dataset:** Amazon Polarity (2000 reviews)
    **Built for:** Amazon ML Summer School 2026
    """)

demo.launch(share=True, debug=False)